# W07A · B · 직접 샘플링과 Gradio 생성 관찰실
2026-10-12 · CPU 기록 분석 / T4 GPU 실제 생성 · 독립 실행

GPU가 있으면 실제 SD 1.5를 내려받아 실행합니다(최초 약2.7GB, 캐시 재사용). CPU에서는 기록 모드를 명시적으로 사용합니다. 모델 오류를 기록 재생으로 자동 대체하지 않습니다. 이전 실습·API 키·개인 파일 업로드가 필요 없습니다.

[강의노트](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/course/notion/w07a/w07a-sampling.md) · [API 입력·출력과 정의](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/src/W07A-API.md) · [실험 기록지](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/course/notion/w07a/w07a-workbook.md)


In [ ]:
import sys, subprocess, os, time
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
PUBLIC_REF='2026-fall-w07a'
try:
    ready=all(version(p)==v for p,v in [('luna-genai','0.1.11'),('diffusers','0.40.0'),('transformers','5.15.1'),('gradio','6.26.0')])
except PackageNotFoundError: ready=False
if not ready:
    if any(p in sys.modules for p in ('luna_genai','diffusers','transformers','gradio')):
        raise RuntimeError('이전 모듈이 남아 있습니다. 런타임 재시작 후 첫 셀부터 실행하세요.')
    package=str(Path('src').resolve()) if Path('src/pyproject.toml').exists() else f'git+https://github.com/lunalab-ai/genAI.git@{PUBLIC_REF}#subdirectory=src'
    subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
torch.set_num_threads(2)
IN_COLAB='COLAB_RELEASE_TAG' in os.environ
print('Python',sys.version.split()[0],'luna-genai',version('luna-genai'),'CUDA',torch.cuda.is_available())


## 1. 실행 모드를 먼저 확인
[`load_sd15(device="cuda")`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/src/luna_genai/stable_diffusion.py#L30)는 고정 revision의 SD1.5와 DDIM, 활성 안전 검사를 준비합니다. 모델은 한 번만 로드하여 아래 함수와 앱에서 공유합니다. GPU를 원하면 Colab 런타임에서 T4를 선택한 뒤 처음부터 실행하세요. [`load_sampling_records`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/src/luna_genai/sampling.py#L188)는 실제 저장 결과를 읽는 별도 함수입니다.


In [ ]:
from luna_genai.stable_diffusion import load_sd15,DEFAULT_PROMPT
from luna_genai.sampling import load_sampling_records,ASSETS,manual_sample,compare_sampling_paths,make_initial_latents
record=load_sampling_records()
RUN_LIVE=IN_COLAB and torch.cuda.is_available()
pipe=load_sd15(device='cuda') if RUN_LIVE else None
print('실행 모드:', '실제 T4 생성' if RUN_LIVE else '저장된 실제 결과 분석')
if pipe is not None: pipe.set_progress_bar_config(disable=True)


## 2. 루프를 읽고 실제로 호출
[`manual_sample(pipe,prompt,negative_prompt="",seed=42,guidance=7.5,steps=20)`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/src/luna_genai/sampling.py#L76)는 실제 512×512 DDIM 생성을 수행합니다. 반환 사전의 image는 최종 PIL 이미지, trace는 단계별 통계, frames는 latent/미리보기, settings는 측정 설정입니다. 모델 가중치를 학습하지 않습니다. 입력이 비거나 seed·steps가 소수면 오류를 냅니다. CFG=1에서는 negative 갈래를 사용하지 않습니다.

[전체 루프와 함수별 정의](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/src/W07A-API.md)를 열고 U-Net → CFG → scheduler 순서를 추적하세요.


In [ ]:
if RUN_LIVE:
    result=manual_sample(pipe,DEFAULT_PROMPT)
    display(result['image']);display(pd.DataFrame(result['trace']))
    print(result['settings'])
else:
    display(Image.open(ASSETS/record['runs'][0]['file']))
    display(pd.DataFrame(record['runs'][0]['trace']))
    print('실제 기록을 읽었습니다. 지금 생성한 결과가 아닙니다.')


### B1 · 갱신과 복원 오류 찾기

`z=z-epsilon`과 `vae.decode(z)`의 문제를 각각 설명하고 올바른 호출을 쓰세요.
힌트: scheduler의 단계 계수와 VAE 설정을 확인합니다.


## 3. 같은 초기 노이즈로 두 구현을 비교
[`compare_sampling_paths`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/src/luna_genai/sampling.py#L161)는 초기 raw latent를 두 경로에 복사하고 동일 DDIM·eta·prompt·negative·dtype·steps로 최종 latent를 비교합니다. 반환 max_abs/mean_abs와 passed를 확인합니다. pipeline의 latent 출력은 수치 검산에만 쓰며, 이미지는 안전 검사한 경로의 결과를 표시합니다. 모델·환경을 바꾸면 이 검증을 다시 해야 합니다.


In [ ]:
parity=compare_sampling_paths(pipe,steps=20) if RUN_LIVE else record['parity']
print('지금 실행한 비교' if RUN_LIVE else '기록된 비교',parity)
assert parity['passed']


### B2 · 한 조건 비교로 다시 설계

seed42/CFG4/20단계와 seed43/CFG7.5/10단계의 차이를 CFG만의 효과라고 말할 수 있을까요? 올바른 두 설정을 제안하세요.
힌트: 바뀐 조건을 세어 보세요.


## 4. negative prompt 하나만 변경
`red`는 빨간색을 지우는 마스크가 아니라 CFG 기준 텍스트입니다. 표면 색, 손잡이, 컵 형태, 배경을 구별해 보세요. 기본 프롬프트에는 red가 들어 있어 조건이 상충하는 실험입니다. 더 높은 CFG나 더 많은 단계가 항상 더 좋은 이미지라는 가정도 하지 않습니다.


In [ ]:
if RUN_LIVE:
    negative_result=manual_sample(pipe,DEFAULT_PROMPT,negative_prompt='red')
    display(negative_result['image']);print(negative_result['settings'])
else:
    negative_run=next(r for r in record['runs'] if r['key']=='negative-red')
    display(Image.open(ASSETS/negative_run['file']));print('기록의 negative prompt:',negative_run['negative_prompt'])


### B3 · 오픈이라는 말 해석

“메타데이터와 모델이 공개되었으므로 모든 원이미지를 무조건 재배포할 수 있다”를 평가하세요. 확인할 문서 두 가지 이상을 쓰세요.
힌트: 코드/가중치/메타데이터/원이미지를 나눕니다.


## 5. 나의 요약 함수부터 작성
아래 기본 함수는 활동 안내만 반환합니다. 이 변수는 문제용이므로 미완성이어도 생성·기록 탭은 계속 동작합니다. 순수 Python 함수가 작동하는지 먼저 확인한 뒤 웹 앱에 전달합니다.


In [ ]:
def make_summary(setting,observation):
    return 'B4: 두 입력을 이용하여 나의 실험 요약 함수를 완성하세요.'
print(make_summary('CFG만 변경','표면 색을 관찰'))


### B4 · 입력 두 개와 출력 하나 연결

make_summary(setting,observation)를 완성하세요. 바꾼 조건/관찰/일반화의 한계를 포함한 문자열을 반환하고, 빈 입력에는 안내를 반환하게 하세요.
힌트: strip(), 문자열 결합부터 확인합니다. 자가 점검: 입력 순서와 반환형을 확인하세요.


## 6. 마지막 웹 애플리케이션
[`build_sampling_app(pipe=pipe, summary_fn=make_summary)`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/src/luna_genai/sampling_app.py#L49)는 준비된 모델과 학생 함수를 받아 앱을 구성합니다. 모델을 다시 내려받지 않습니다. 첫 탭의 기록과 GPU가 연결된 둘째 탭의 실제 생성을 구별하세요. [`sampling_record_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/src/luna_genai/sampling_app.py#L20)는 실제 기록을 읽어 callback 출력 계약을 검사하는 데도 쓰입니다.

Colab 출력의 공유 URL을 새 탭에서 열고 기록 선택 → 버튼 → gallery → 설정 JSON 순서로 확인하세요. 생성 탭에서는 seed를 고정하고 negative 또는 CFG 하나만 변경합니다. 공유 링크는 런타임 종료 뒤 사용할 수 없습니다.


In [ ]:
from luna_genai.sampling_app import build_sampling_app,sampling_record_view
app=build_sampling_app(pipe=pipe,summary_fn=make_summary)
callback_output=sampling_record_view('negative-red')
if IN_COLAB:
    app.queue().launch(share=True,inline=True,prevent_thread_lock=True,show_error=True)


### B5 · 출구 설명

문장부터 이미지까지 여섯 단계 이내로 U-Net·CFG·scheduler·VAE의 역할을 설명하세요. 고정할 조건 두 가지와 관찰의 한계 한 가지도 적으세요.
힌트: 화살표마다 동사를 붙입니다.


In [ ]:
assert parity['passed'] and parity['max_abs']<=.002
assert callback_output[0].size==(512,512) and len(callback_output[1])==5
assert callback_output[2]['negative_prompt']=='red'
assert isinstance(make_summary('조건','관찰'),str)
assert Path(callback_output[3]).is_file()
assert len(app.config['dependencies'])>=4
if RUN_LIVE:
    assert torch.isfinite(result['final_latent']).all()
    assert result['settings']['safety_checked'] and negative_result['settings']['safety_checked']
    assert result['settings']['initial_sha256']==negative_result['settings']['initial_sha256']
    print('실제 GPU 생성 및 pipeline 정합성 검증 통과')
else:
    print('CPU 기록 분석 검증 통과 · 이번 실행에서는 SD 생성하지 않음')
print('W07A B 검증 통과 · 입력/출력/모드/앱 callback 확인')
if not IN_COLAB: app.close()
